In [ ]:
# In[1]:

import pandas as pd

# Load CSVs into DataFrames (stored as kernel variables for later steps)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes but keep original integer timestamp columns
for df in (metric_df, trace_df, log_df, error_logs_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = df['timestamp'].astype('int64')  # ensure ints
        df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    else:
        # if missing timestamp column, create empty placeholders
        df['timestamp'] = pd.Series(dtype='int64')
        df['timestamp_dt'] = pd.Series(dtype='datetime64[ns, UTC]')

# --- metric.csv summary ---
metric_rows = metric_df.shape[0]
metric_cmdbs = pd.Series(metric_df['cmdb_id'].dropna().unique(), name='cmdb_id')
if 'kpi_name' in metric_df.columns:
    metric_kpi_counts = (
        metric_df['kpi_name']
        .value_counts()
        .rename_axis('kpi_name')
        .reset_index(name='count')
        .head(50)
    )
else:
    metric_kpi_counts = pd.DataFrame(columns=['kpi_name', 'count'])
metric_min_ts = int(metric_df['timestamp'].min()) if metric_rows>0 else None
metric_max_ts = int(metric_df['timestamp'].max()) if metric_rows>0 else None

# --- trace.csv summary ---
trace_rows = trace_df.shape[0]
trace_cmdbs = pd.Series(trace_df['cmdb_id'].dropna().unique(), name='cmdb_id')
if 'trace_name' in trace_df.columns:
    trace_name_counts = (
        trace_df['trace_name']
        .value_counts()
        .rename_axis('trace_name')
        .reset_index(name='count')
        .head(50)
    )
else:
    trace_name_counts = pd.DataFrame(columns=['trace_name', 'count'])
trace_min_ts = int(trace_df['timestamp'].min()) if trace_rows>0 else None
trace_max_ts = int(trace_df['timestamp'].max()) if trace_rows>0 else None

# --- log.csv summary ---
log_rows = log_df.shape[0]
log_cmdbs = pd.Series(log_df['cmdb_id'].dropna().unique(), name='cmdb_id')
if 'log_name' in log_df.columns:
    log_name_counts = (
        log_df['log_name']
        .value_counts()
        .rename_axis('log_name')
        .reset_index(name='count')
        .head(50)
    )
else:
    log_name_counts = pd.DataFrame(columns=['log_name', 'count'])
log_min_ts = int(log_df['timestamp'].min()) if log_rows>0 else None
log_max_ts = int(log_df['timestamp'].max()) if log_rows>0 else None

# --- error_logs.csv summary ---
error_logs_rows = error_logs_df.shape[0]
error_logs_cmdbs = pd.Series(error_logs_df['cmdb_id'].dropna().unique(), name='cmdb_id')
# create message prefix (first 120 chars) to summarize distinct messages
if 'message' in error_logs_df.columns:
    msg_prefix = error_logs_df['message'].fillna('').astype(str).str.slice(0, 120)
    error_msg_prefix_counts = (
        msg_prefix.value_counts()
        .rename_axis('message_prefix')
        .reset_index(name='count')
        .head(10)
    )
else:
    error_msg_prefix_counts = pd.DataFrame(columns=['message_prefix', 'count'])
error_logs_min_ts = int(error_logs_df['timestamp'].min()) if error_logs_rows>0 else None
error_logs_max_ts = int(error_logs_df['timestamp'].max()) if error_logs_rows>0 else None

# Compact summary tables for display
metric_summary = pd.DataFrame({
    'rows': [metric_rows],
    'distinct_cmdb_count': [metric_cmdbs.size],
    'earliest_ts': [metric_min_ts],
    'latest_ts': [metric_max_ts]
})

trace_summary = pd.DataFrame({
    'rows': [trace_rows],
    'distinct_cmdb_count': [trace_cmdbs.size],
    'earliest_ts': [trace_min_ts],
    'latest_ts': [trace_max_ts]
})

log_summary = pd.DataFrame({
    'rows': [log_rows],
    'distinct_cmdb_count': [log_cmdbs.size],
    'earliest_ts': [log_min_ts],
    'latest_ts': [log_max_ts]
})

error_logs_summary = pd.DataFrame({
    'rows': [error_logs_rows],
    'distinct_cmdb_count': [error_logs_cmdbs.size],
    'earliest_ts': [error_logs_min_ts],
    'latest_ts': [error_logs_max_ts]
})

# Final compact outputs (these variables will be returned/displayed by the IPython kernel)
metric_df, metric_summary, metric_cmdbs, metric_kpi_counts, metric_min_ts, metric_max_ts, \
trace_df, trace_summary, trace_cmdbs, trace_name_counts, trace_min_ts, trace_max_ts, \
log_df, log_summary, log_cmdbs, log_name_counts, log_min_ts, log_max_ts, \
error_logs_df, error_logs_summary, error_logs_cmdbs, error_msg_prefix_counts, error_logs_min_ts, error_logs_max_ts

```
Out[1]:
```
# Summarize execution results in plain English (compact)
def top_pairs(df, name_col, count_col='count', n=8):
    if df is None or df.empty:
        return "none"
    pairs = []
    for r in df.head(n).itertuples(index=False):
        pairs.append(f"{getattr(r, name_col)}: {getattr(r, count_col)}")
    return ", ".join(pairs)

# Extract summaries (these variables were created in previous step)
metric_rows = int(metric_summary.at[0, 'rows'])
metric_cmdb_count = int(metric_summary.at[0, 'distinct_cmdb_count'])
metric_min_ts = int(metric_summary.at[0, 'earliest_ts'])
metric_max_ts = int(metric_summary.at[0, 'latest_ts'])
metric_top_kpis = top_pairs(metric_kpi_counts, 'kpi_name')

trace_rows = int(trace_summary.at[0, 'rows'])
trace_cmdb_count = int(trace_summary.at[0, 'distinct_cmdb_count'])
trace_min_ts = int(trace_summary.at[0, 'earliest_ts'])
trace_max_ts = int(trace_summary.at[0, 'latest_ts'])
trace_top = top_pairs(trace_name_counts, 'trace_name')

log_rows = int(log_summary.at[0, 'rows'])
log_cmdb_count = int(log_summary.at[0, 'distinct_cmdb_count'])
log_min_ts = int(log_summary.at[0, 'earliest_ts'])
log_max_ts = int(log_summary.at[0, 'latest_ts'])
log_top = top_pairs(log_name_counts, 'log_name')

err_rows = int(error_logs_summary.at[0, 'rows'])
err_cmdb_count = int(error_logs_summary.at[0, 'distinct_cmdb_count'])
err_min = error_logs_summary.at[0, 'earliest_ts']
err_max = error_logs_summary.at[0, 'latest_ts']

summary = (
    f"metric.csv: {metric_rows} rows, {metric_cmdb_count} distinct cmdb_ids. "
    f"Top KPIs (sample): {metric_top_kpis}. "
    f"Timestamps min={metric_min_ts}, max={metric_max_ts}.\n"
    f"trace.csv: {trace_rows} rows, {trace_cmdb_count} distinct cmdb_ids. "
    f"Top trace_names (sample): {trace_top}. "
    f"Timestamps min={trace_min_ts}, max={trace_max_ts}.\n"
    f"log.csv: {log_rows} rows, {log_cmdb_count} distinct cmdb_ids. "
    f"Top log_names: {log_top}. "
    f"Timestamps min={log_min_ts}, max={log_max_ts}.\n"
    f"error_logs.csv: {err_rows} rows (empty). distinct cmdb_ids={err_cmdb_count}. "
    f"earliest_ts={err_min}, latest_ts={err_max}."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(       timestamp                      cmdb_id    kpi_name         value              timestamp_dt
0     1705951500  ts-admin-basic-info-service         cpu  4.857230e-01 2024-01-22 19:25:00+00:00
1     1705951500  ts-admin-basic-info-service      diskio  5.411453e+04 2024-01-22 19:25:00+00:00
2     1705951500  ts-admin-basic-info-service  latency-50  1.358885e-02 2024-01-22 19:25:00+00:00
3     1705951500  ts-admin-basic-info-service  latency-90  2.391917e-02 2024-01-22 19:25:00+00:00
4     1705951500  ts-admin-basic-info-service         mem  2.364662e+08 2024-01-22 19:25:00+00:00
...          ...                          ...         ...           ...                       ...
9345  1705952940             ts-voucher-mysql      socket  2.000000e+00 2024-01-22 19:49:00+00:00
9346  1705952940           ts-voucher-service         cpu  3.161806e-02 2024-01-22 19:49:00+00:00
9347  1705952940           ts-voucher-service      diskio  0.000000e+00 2024-01-22 19:49:00+00:00
9348  1705952940           ts-voucher-service         mem  3.801088e+07 2024-01-22 19:49:00+00:00
9349  1705952940           ts-voucher-service      socket  4.000000e+00 2024-01-22 19:49:00+00:00

[9350 rows x 5 columns],    rows  distinct_cmdb_count  earliest_ts   latest_ts
0  9350                   68   1705951500  1705952940, 0      ts-admin-basic-info-service
1           ts-admin-order-service
2           ts-admin-route-service
3          ts-admin-travel-service
4            ts-admin-user-service
                  ...             
63                   ts-user-mongo
64                 ts-user-service
65    ts-verification-code-service
66                ts-voucher-mysql
67              ts-voucher-service
Name: cmdb_id, Length: 68, dtype: object,      kpi_name  count
0         cpu   1700
1         mem   1700
2      socket   1700
3      diskio   1675
4  latency-90    700
5  latency-50    700
6    workload    700
7       error    475, 1705951500, 1705952940,         timestamp             cmdb_id                                         trace_name       value              timestamp_dt
0      1705951500                root  trace.to_ts-admin-basic-info-service.duration_...    0.010356 2024-01-22 19:25:00+00:00
1      1705951500                root  trace.to_ts-admin-basic-info-service.duration_p95    0.015045 2024-01-22 19:25:00+00:00
2      1705951500                root    trace.to_ts-admin-basic-info-service.error_rate    0.000000 2024-01-22 19:25:00+00:00
3      1705951500                root     trace.to_ts-admin-basic-info-service.row_count   10.000000 2024-01-22 19:25:00+00:00
4      1705951500                root     trace.to_ts-admin-travel-service.duration_mean    0.112112 2024-01-22 19:25:00+00:00
...           ...                 ...                                                ...         ...                       ...
13083  1705952640  ts-travel2-service                trace.to_ts-train-service.row_count   89.000000 2024-01-22 19:44:00+00:00
13084  1705952640  ts-travel2-service          trace.to_ts-travel2-service.duration_mean    0.011428 2024-01-22 19:44:00+00:00
13085  1705952640  ts-travel2-service           trace.to_ts-travel2-service.duration_p95    0.054317 2024-01-22 19:44:00+00:00
13086  1705952640  ts-travel2-service             trace.to_ts-travel2-service.error_rate    0.000000 2024-01-22 19:44:00+00:00
13087  1705952640  ts-travel2-service              trace.to_ts-travel2-service.row_count  858.000000 2024-01-22 19:44:00+00:00

[13088 rows x 5 columns],     rows  distinct_cmdb_count  earliest_ts   latest_ts
0  13088                   28   1705951500  1705952640, 0                            root
1     ts-admin-basic-info-service
2         ts-admin-travel-service
3            ts-assurance-service
4                 ts-auth-service
                 ...             
23          ts-ticketinfo-service
24               ts-train-service
25              ts-travel-service
26             ts-travel2-service
27                ts-user-service
Name: cmdb_id, Length: 28, dtype: object,                                       trace_name  count
0                      trace.from_root.row_count    439
1                     trace.from_root.error_rate    439
2                   trace.from_root.duration_p95    439
3                  trace.from_root.duration_mean    439
4   trace.from_ts-preserve-service.duration_mean    150
..                                           ...    ...
45     trace.to_ts-ticketinfo-service.error_rate     88
46   trace.to_ts-ticketinfo-service.duration_p95     88
47  trace.to_ts-ticketinfo-service.duration_mean     88
48           trace.to_ts-train-service.row_count     82
49          trace.to_ts-train-service.error_rate     82

[50 rows x 2 columns], 1705951500, 1705952640,        timestamp                      cmdb_id         log_name  value              timestamp_dt
0     1705951500  ts-admin-basic-info-service  log.error_count      0 2024-01-22 19:25:00+00:00
1     1705951500  ts-admin-basic-info-service    log.row_count     16 2024-01-22 19:25:00+00:00
2     1705951500      ts-admin-travel-service  log.error_count      0 2024-01-22 19:25:00+00:00
3     1705951500      ts-admin-travel-service    log.row_count     15 2024-01-22 19:25:00+00:00
4     1705951500           ts-assurance-mongo  log.error_count      0 2024-01-22 19:25:00+00:00
...          ...                          ...              ...    ...                       ...
1311  1705952640            ts-travel-service    log.row_count    598 2024-01-22 19:44:00+00:00
1312  1705952640           ts-travel2-service  log.error_count      0 2024-01-22 19:44:00+00:00
1313  1705952640           ts-travel2-service    log.row_count   1134 2024-01-22 19:44:00+00:00
1314  1705952640              ts-ui-dashboard  log.error_count      0 2024-01-22 19:44:00+00:00
1315  1705952640              ts-ui-dashboard    log.row_count    133 2024-01-22 19:44:00+00:00

[1316 rows x 5 columns],    rows  distinct_cmdb_count  earliest_ts   latest_ts
0  1316                   45   1705951500  1705952640, 0     ts-admin-basic-info-service
1         ts-admin-travel-service
2              ts-assurance-mongo
3            ts-assurance-service
4                   ts-auth-mongo
                 ...             
40              ts-travel-service
41               ts-travel2-mongo
42             ts-travel2-service
43                ts-ui-dashboard
44                  ts-user-mongo
Name: cmdb_id, Length: 45, dtype: object,           log_name  count
0  log.error_count    658
1    log.row_count    658, 1705951500, 1705952640, Empty DataFrame
Columns: [timestamp, cmdb_id, message, error, timestamp_dt]
Index: [],    rows  distinct_cmdb_count earliest_ts latest_ts
0     0                    0        None      None, Series([], Name: cmdb_id, dtype: object), Empty DataFrame
Columns: [message_prefix, count]
Index: [], None, None)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse metric_df from kernel

# 1) Global P95 per (cmdb_id, kpi_name) using the entire metric_df
comp_kpi_thresholds = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'p95_threshold'})
)

# 2) Filter metric_df to the incident window (inclusive) using timestamp_dt (UTC)
start_ts = pd.to_datetime('2024-01-22 19:22:25', utc=True)
end_ts   = pd.to_datetime('2024-01-22 19:52:25', utc=True)
metric_window = metric_df[(metric_df['timestamp_dt'] >= start_ts) & (metric_df['timestamp_dt'] <= end_ts)].copy()

# 3) Join windowed data with thresholds so each row knows its global P95
mw = metric_window.merge(comp_kpi_thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Mark exceedances (value > global P95)
mw['exceed'] = mw['value'] > mw['p95_threshold']

# Compute basic aggregates per (cmdb_id, kpi_name)
aggregates = (
    mw
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(
        count_in_window = ('value', 'size'),
        count_exceed    = ('exceed', 'sum'),
        max_value       = ('value', 'max'),
    )
    .reset_index()
)

# Compute earliest exceed timestamp per group (if any)
earliest_exceed = (
    mw.loc[mw['exceed']]
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['timestamp_dt']
    .min()
    .reset_index()
    .rename(columns={'timestamp_dt': 'earliest_exceed_ts_dt'})
)

# Merge earliest exceed into aggregates
agg_merged = aggregates.merge(earliest_exceed, on=['cmdb_id', 'kpi_name'], how='left')

# Compute exceed_ratio and format earliest_exceed_ts as ISO UTC strings
agg_merged['exceed_ratio'] = agg_merged['count_exceed'] / agg_merged['count_in_window']
agg_merged['earliest_exceed_ts'] = agg_merged['earliest_exceed_ts_dt'].apply(
    lambda dt: dt.isoformat() if pd.notnull(dt) else None
)

# Final selected columns and rename for requested names
agg_final = agg_merged[[
    'cmdb_id', 'kpi_name', 'count_in_window', 'count_exceed',
    'earliest_exceed_ts', 'max_value', 'exceed_ratio'
]].copy()

# 4) Compact table sorted by count_exceed descending (max 20 rows)
agg_display = agg_final.sort_values(by='count_exceed', ascending=False).head(20)

# Total count of distinct cmdb_ids that have at least one exceedance, and the list (max 50)
exceeding_cmdbs = sorted(agg_final.loc[agg_final['count_exceed'] > 0, 'cmdb_id'].dropna().unique())
total_exceed_cmdb_count = len(exceeding_cmdbs)
exceed_cmdb_list = pd.Series(exceeding_cmdbs[:50], name='cmdb_id')

# Provide shapes of stored full tables for confirmation
comp_kpi_thresholds_shape = comp_kpi_thresholds.shape
metric_window_shape = metric_window.shape

# Return compact results (kernel will display these variables)
agg_display, total_exceed_cmdb_count, exceed_cmdb_list, comp_kpi_thresholds_shape, metric_window_shape

```
Out[2]:
```
summary = (
    f"Analysis summary (plain English):\n\n"
    f"- I computed global P95 thresholds per (cmdb_id, kpi_name) across the entire metric series and stored them in 'comp_kpi_thresholds' "
    f"(shape={comp_kpi_thresholds_shape}).\n"
    f"- I filtered metric data to the incident window 2024-01-22 19:22:25 UTC through 2024-01-22 19:52:25 UTC and stored it in 'metric_window' "
    f"(shape={metric_window_shape}).\n\n"
    f"- Per-(cmdb_id,kpi_name) aggregates within the window were produced with these columns: "
    f"count_in_window, count_exceed, earliest_exceed_ts (ISO UTC), max_value, exceed_ratio.\n\n"
    f"- Compact findings (top examples): many services show 2 points exceeding their global P95 out of 25 samples in the window "
    f"(count_in_window=25, count_exceed=2, exceed_ratio=0.08). Examples include:\n"
    f"  • ts-voucher-service (cpu) — earliest exceed at 2024-01-22T19:25:00+00:00, max_value ~4.1055e-02\n"
    f"  • ts-admin-basic-info-service (cpu, diskio, latency-50, latency-90, mem, socket, workload) — various earliest exceed times (e.g. 19:39, 19:27, 19:25 UTC), max values shown in the aggregate table.\n"
    f"  • several other admin, order, route, travel services show the same pattern (2 exceedances in the 25-point window).\n\n"
    f"- In total, {total_exceed_cmdb_count} distinct cmdb_id(s) had at least one metric value exceeding their global P95 within the incident window. "
    f"(A sample/list of up to 50 such cmdb_ids is available in the variable 'exceed_cmdb_list'.)\n\n"
    f"- Stored outputs for follow-up: 'comp_kpi_thresholds' (global P95s), 'metric_window' (filtered telemetry), "
    f"and the per-(cmdb_id,kpi_name) aggregate table (saved in the kernel as 'agg_final' / 'agg_display').\n\n"
    f"Next steps (suggested): inspect the aggregate table 'agg_display' for the full top offenders, then drill into raw 'metric_window' rows "
    f"for those (cmdb_id,kpi_name) around the earliest_exceed_ts to understand context and correlate with traces/logs."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                         cmdb_id    kpi_name  count_in_window  count_exceed         earliest_exceed_ts     max_value  exceed_ratio
370           ts-voucher-service         cpu               25             2  2024-01-22T19:25:00+00:00  4.105519e-02          0.08
0    ts-admin-basic-info-service         cpu               25             2  2024-01-22T19:39:00+00:00  7.025239e-01          0.08
1    ts-admin-basic-info-service      diskio               25             2  2024-01-22T19:27:00+00:00  1.310452e+05          0.08
2    ts-admin-basic-info-service  latency-50               25             2  2024-01-22T19:27:00+00:00  1.582209e-02          0.08
3    ts-admin-basic-info-service  latency-90               25             2  2024-01-22T19:25:00+00:00  2.398193e-02          0.08
4    ts-admin-basic-info-service         mem               25             2  2024-01-22T19:45:00+00:00  2.366478e+08          0.08
5    ts-admin-basic-info-service      socket               25             2  2024-01-22T19:38:00+00:00  7.016667e+00          0.08
6    ts-admin-basic-info-service    workload               25             2  2024-01-22T19:43:00+00:00  7.965167e-01          0.08
7         ts-admin-order-service         cpu               25             2  2024-01-22T19:30:00+00:00  1.044670e-01          0.08
8         ts-admin-order-service      diskio               25             2  2024-01-22T19:29:00+00:00  6.601777e+04          0.08
9         ts-admin-order-service         mem               25             2  2024-01-22T19:35:00+00:00  1.925802e+08          0.08
10        ts-admin-order-service      socket               25             2  2024-01-22T19:25:00+00:00  3.716667e+00          0.08
11        ts-admin-route-service         cpu               25             2  2024-01-22T19:28:00+00:00  1.009753e-01          0.08
12        ts-admin-route-service      diskio               25             2  2024-01-22T19:26:00+00:00  1.264701e+05          0.08
13        ts-admin-route-service         mem               25             2  2024-01-22T19:39:00+00:00  1.890953e+08          0.08
327            ts-travel-service  latency-90               25             2  2024-01-22T19:46:00+00:00  2.727539e+00          0.08
328            ts-travel-service         mem               25             2  2024-01-22T19:48:00+00:00  2.924702e+08          0.08
329            ts-travel-service      socket               25             2  2024-01-22T19:38:00+00:00  2.530000e+01          0.08
330            ts-travel-service    workload               25             2  2024-01-22T19:29:00+00:00  1.087333e+01          0.08
331             ts-travel2-mongo         cpu               25             2  2024-01-22T19:27:00+00:00  8.682759e-01          0.08, 68, 0     ts-admin-basic-info-service
1          ts-admin-order-service
2          ts-admin-route-service
3         ts-admin-travel-service
4           ts-admin-user-service
                 ...             
45          ts-route-plan-service
46               ts-route-service
47                ts-seat-service
48              ts-security-mongo
49            ts-security-service
Name: cmdb_id, Length: 50, dtype: object, (374, 3), (9350, 5))```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing kernel variables: comp_kpi_thresholds, metric_window

# 1) Join metric_window with comp_kpi_thresholds to get P95 per row
mw = metric_window.merge(comp_kpi_thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Ensure timestamp ordering
mw = mw.sort_values(['cmdb_id', 'kpi_name', 'timestamp_dt']).reset_index(drop=True)

# 2) Compute exceed boolean where value > P95
mw['exceed'] = mw['value'] > mw['p95_threshold']

# 3) Identify consecutive True runs per (cmdb_id, kpi_name)
# Create segment ids that increment whenever exceed is False (so consecutive Trues share same segment)
def assign_segments(df):
    # df is already sorted by timestamp_dt
    df = df.copy()
    # (~exceed).cumsum() will create same segment id for consecutive Trues
    df['seg'] = (~df['exceed']).cumsum()
    return df

mw_seg = mw.groupby(['cmdb_id', 'kpi_name'], group_keys=False).apply(assign_segments).reset_index(drop=True)

# Aggregate segments to runs; only keep segments where exceed==True
seg_groups = (
    mw_seg
    .loc[mw_seg['exceed']]  # only rows that exceed
    .groupby(['cmdb_id', 'kpi_name', 'seg'], dropna=False)
    .agg(
        run_start_ts_dt = ('timestamp_dt', 'min'),
        run_end_ts_dt   = ('timestamp_dt', 'max'),
        run_length      = ('timestamp_dt', 'count'),
        max_value_in_run= ('value', 'max'),
        P95             = ('p95_threshold', 'first')
    )
    .reset_index()
)

# Compute severity_ratio handling P95==0
seg_groups['severity_ratio'] = seg_groups.apply(
    lambda r: (r['max_value_in_run'] / r['P95']) if pd.notnull(r['P95']) and r['P95'] != 0 else np.nan,
    axis=1
)

# Format ISO UTC strings for start/end
seg_groups['run_start_ts'] = seg_groups['run_start_ts_dt'].apply(lambda dt: dt.isoformat() if pd.notnull(dt) else None)
seg_groups['run_end_ts']   = seg_groups['run_end_ts_dt'].apply(lambda dt: dt.isoformat() if pd.notnull(dt) else None)

# Select and order columns as requested
metric_fault_runs = seg_groups[[
    'cmdb_id', 'kpi_name', 'run_start_ts', 'run_end_ts',
    'run_length', 'max_value_in_run', 'P95', 'severity_ratio',
    'run_start_ts_dt', 'run_end_ts_dt', 'seg'  # keep extras in full table for traceability
]].copy()

# 4) Filter to runs with run_length >= 3
metric_fault_runs = metric_fault_runs.loc[metric_fault_runs['run_length'] >= 3].reset_index(drop=True)

# 5) Prepare compact summary: sorted by run_length desc then severity_ratio desc, limit 20 rows
metric_fault_runs_display = (
    metric_fault_runs
    .sort_values(['run_length', 'severity_ratio'], ascending=[False, False])
    .loc[:, ['cmdb_id', 'kpi_name', 'run_start_ts', 'run_end_ts', 'run_length', 'max_value_in_run', 'P95', 'severity_ratio']]
    .head(20)
)

# Total number of runs found
total_runs_found = metric_fault_runs.shape[0]

# Return compact outputs (kernel will display these variables)
metric_fault_runs_display, total_runs_found, metric_fault_runs.shape, metric_fault_runs.head(3)

```
Out[3]:
```
summary = (
    f"No fault runs detected: {total_runs_found} runs with >=3 consecutive points above the global P95 were found in the incident window.\n\n"
    "Context & recommendation:\n"
    "- Earlier analysis showed occasional exceedances (mostly 2 points each) across many services, but none persisted for 3+ consecutive minutes.\n"
    "- This suggests short spikes rather than sustained anomalies. To investigate further, consider:\n"
    "  1) Lowering the run_length threshold to 2 to capture short bursts.\n"
    "  2) Inspecting 'metric_window' for the cmdb_ids with any exceedances and their timestamps.\n"
    "  3) Correlating those timestamps with traces/logs (variables: 'metric_window', 'comp_kpi_thresholds', 'metric_fault_runs').\n\n"
    "Stored data you can use next: comp_kpi_thresholds (global P95s), metric_window (filtered metrics), and metric_fault_runs (detected runs, empty for current threshold)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, kpi_name, run_start_ts, run_end_ts, run_length, max_value_in_run, P95, severity_ratio]
Index: [], 0, (0, 11), Empty DataFrame
Columns: [cmdb_id, kpi_name, run_start_ts, run_end_ts, run_length, max_value_in_run, P95, severity_ratio, run_start_ts_dt, run_end_ts_dt, seg]
Index: [])```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse metric_df from kernel

# 1) Candidate services
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# 2) Global P95 per (cmdb_id, kpi_name) but only for candidate cmdb_ids (thresholds computed from full metric_df)
candidate_thresholds = (
    metric_df[metric_df['cmdb_id'].isin(candidates)]
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'p95_threshold'})
)

# 3) Filter metric_df to incident window and candidate services
start_ts = pd.to_datetime('2024-01-22 19:22:25', utc=True)
end_ts   = pd.to_datetime('2024-01-22 19:52:25', utc=True)
candidate_metric_window = metric_df[
    (metric_df['timestamp_dt'] >= start_ts) &
    (metric_df['timestamp_dt'] <= end_ts) &
    (metric_df['cmdb_id'].isin(candidates))
].copy()

# 4) Join with candidate_thresholds to get P95 per row, mark exceed (value > P95)
cmw = candidate_metric_window.merge(candidate_thresholds, on=['cmdb_id', 'kpi_name'], how='left')
cmw = cmw.sort_values(['cmdb_id', 'kpi_name', 'timestamp_dt']).reset_index(drop=True)
cmw['exceed'] = cmw['value'] > cmw['p95_threshold']

# 4-5) Detect consecutive True runs per (cmdb_id,kpi_name); keep runs with run_length >= 2
def assign_seg(df):
    df = df.copy()
    # segment id increments on False rows; consecutive True rows share same seg id
    df['seg'] = (~df['exceed']).cumsum()
    return df

cmw_seg = cmw.groupby(['cmdb_id', 'kpi_name'], group_keys=False).apply(assign_seg).reset_index(drop=True)

runs = (
    cmw_seg.loc[cmw_seg['exceed']]
    .groupby(['cmdb_id', 'kpi_name', 'seg'], dropna=False)
    .agg(
        run_start_ts_dt = ('timestamp_dt', 'min'),
        run_end_ts_dt   = ('timestamp_dt', 'max'),
        run_length      = ('timestamp_dt', 'count'),
        max_value_in_run= ('value', 'max'),
        P95             = ('p95_threshold', 'first')
    )
    .reset_index(drop=False)
)

# Compute severity_ratio, handle P95 == 0
runs['severity_ratio'] = np.where(runs['P95'].fillna(0) == 0, np.nan, runs['max_value_in_run'] / runs['P95'])

# Format ISO strings
runs['run_start_ts'] = runs['run_start_ts_dt'].apply(lambda dt: dt.isoformat() if pd.notnull(dt) else None)
runs['run_end_ts']   = runs['run_end_ts_dt'].apply(lambda dt: dt.isoformat() if pd.notnull(dt) else None)

# Select required columns
candidate_metric_fault_runs = runs[[
    'cmdb_id', 'kpi_name', 'run_start_ts', 'run_end_ts',
    'run_length', 'max_value_in_run', 'P95', 'severity_ratio',
    'run_start_ts_dt', 'run_end_ts_dt', 'seg'
]].copy()

# Filter to run_length >= 2
candidate_metric_fault_runs = candidate_metric_fault_runs.loc[candidate_metric_fault_runs['run_length'] >= 2].reset_index(drop=True)

# 6) Compact table sorted by run_length desc then severity_ratio desc (limit 20 rows)
runs_display = (
    candidate_metric_fault_runs
    .sort_values(['run_length', 'severity_ratio'], ascending=[False, False])
    .loc[:, ['cmdb_id', 'kpi_name', 'run_start_ts', 'run_end_ts', 'run_length', 'max_value_in_run', 'P95', 'severity_ratio']]
    .head(20)
)

# Total number of runs and distinct candidate cmdb_ids that had runs (max 10)
total_runs_found = int(candidate_metric_fault_runs.shape[0])
candidate_run_cmdbs = pd.Series(sorted(candidate_metric_fault_runs['cmdb_id'].dropna().unique())[:10], name='cmdb_id')

# Return compact results (these variables are stored in kernel for next steps)
runs_display, total_runs_found, candidate_run_cmdbs, candidate_thresholds.shape, candidate_metric_window.shape, candidate_metric_fault_runs.shape

```
Out[4]:
```
summary = (
    "Summary of candidate-service fault-run detection (plain English):\n\n"
    "- Candidates: ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service.\n"
    "- I computed global P95s for these services and inspected their metrics in the incident window 2024-01-22 19:22:25 → 19:52:25 UTC.\n\n"
    "- Detection results:\n"
    "  • Total runs found (>=2 consecutive exceedances): 13 runs.\n"
    "  • Distinct candidate services with runs (5): ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service.\n"
    "  • All detected runs are short (run_length = 2 points each).\n\n"
    "- Top notable runs (examples):\n"
    "  1) ts-order-service — diskio, 2024-01-22T19:25:00 → 19:26:00, run_length=2, max=1.9881e+05, P95=7.4127e+04, severity_ratio≈2.68\n"
    "  2) ts-travel-service — diskio, 2024-01-22T19:25:00 → 19:26:00, run_length=2, max=1.37295e+05, P95=6.9772e+04, severity_ratio≈1.97\n"
    "  3) ts-travel-service — latency-90, 2024-01-22T19:46:00 → 19:47:00, run_length=2, severity_ratio≈1.12\n"
    "  4) ts-route-service / ts-train-service — several latency and cpu/workload runs with mild severity (~1.03–1.11).\n"
    "  5) ts-auth-service / ts-travel-service — single mem runs with severity ≈1.00 (very marginal).\n\n"
    "- Interpretation & recommended next steps:\n"
    "  • The strongest signals are disk I/O spikes on ts-order-service and ts-travel-service (severity_ratio >1.9). These merit immediate investigation.\n"
    "  • Most runs are exactly 2 consecutive minutes (short bursts). If you want to capture shorter bursts, this analysis used run_length>=2; for sustained faults raise threshold to >=3.\n"
    "  • Next actions: examine raw metric rows in 'candidate_metric_window' and correlate with traces/logs around the listed run_start_ts for the top offenders (especially diskio on order/travel services).\n\n"
    "Stored kernel variables you can use: candidate_thresholds (global P95s), candidate_metric_window (filtered metrics), candidate_metric_fault_runs (detected runs).\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name               run_start_ts                 run_end_ts  run_length  max_value_in_run           P95  severity_ratio
1    ts-order-service      diskio  2024-01-22T19:25:00+00:00  2024-01-22T19:26:00+00:00           2      1.988097e+05  7.412724e+04        2.682006
9   ts-travel-service      diskio  2024-01-22T19:25:00+00:00  2024-01-22T19:26:00+00:00           2      1.372950e+05  6.977201e+04        1.967767
10  ts-travel-service  latency-90  2024-01-22T19:46:00+00:00  2024-01-22T19:47:00+00:00           2      2.727539e+00  2.442698e+00        1.116609
5    ts-route-service  latency-50  2024-01-22T19:27:00+00:00  2024-01-22T19:28:00+00:00           2      4.653679e-03  4.192202e-03        1.110080
2    ts-order-service  latency-50  2024-01-22T19:46:00+00:00  2024-01-22T19:47:00+00:00           2      4.822902e-01  4.375339e-01        1.102292
12  ts-travel-service    workload  2024-01-22T19:29:00+00:00  2024-01-22T19:30:00+00:00           2      1.087333e+01  1.002374e+01        1.084758
7    ts-train-service  latency-50  2024-01-22T19:27:00+00:00  2024-01-22T19:28:00+00:00           2      7.180282e-03  6.848149e-03        1.048500
6    ts-route-service  latency-90  2024-01-22T19:27:00+00:00  2024-01-22T19:28:00+00:00           2      9.046846e-03  8.754114e-03        1.033439
4    ts-route-service         cpu  2024-01-22T19:28:00+00:00  2024-01-22T19:29:00+00:00           2      4.677445e+00  4.539494e+00        1.030389
3    ts-order-service    workload  2024-01-22T19:29:00+00:00  2024-01-22T19:30:00+00:00           2      7.217683e+00  7.126953e+00        1.012731
8    ts-train-service      socket  2024-01-22T19:36:00+00:00  2024-01-22T19:37:00+00:00           2      1.876667e+01  1.867333e+01        1.004998
11  ts-travel-service         mem  2024-01-22T19:48:00+00:00  2024-01-22T19:49:00+00:00           2      2.924702e+08  2.924484e+08        1.000074
0     ts-auth-service         mem  2024-01-22T19:48:00+00:00  2024-01-22T19:49:00+00:00           2      2.441940e+08  2.441925e+08        1.000006, 13, 0      ts-auth-service
1     ts-order-service
2     ts-route-service
3     ts-train-service
4    ts-travel-service
Name: cmdb_id, dtype: object, (39, 3), (975, 5), (13, 11))```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse trace_df from kernel

# 1) Global P95 per trace_name across entire trace_df
trace_name_thresholds = (
    trace_df
    .groupby('trace_name', dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'p95_threshold'})
)

# 2) Filter trace_df to the incident window (UTC)
start_ts = pd.to_datetime('2024-01-22 19:22:25', utc=True)
end_ts   = pd.to_datetime('2024-01-22 19:52:25', utc=True)
trace_window = trace_df[(trace_df['timestamp_dt'] >= start_ts) & (trace_df['timestamp_dt'] <= end_ts)].copy()

# 3) Select rows where trace_name calls any candidate service
pattern = r"(to_ts-order-service|to_ts-travel-service|to_ts-train-service|to_ts-route-service|to_ts-auth-service)"
candidate_trace_window = trace_window[trace_window['trace_name'].str.contains(pattern, na=False)].copy()

# 4) Join with thresholds and compute exceed boolean
ctw = candidate_trace_window.merge(trace_name_thresholds, on='trace_name', how='left')
ctw['exceed'] = ctw['value'] > ctw['p95_threshold']

# 4-5) Aggregate per trace_name
agg = (
    ctw
    .groupby('trace_name', dropna=False)
    .agg(
        count_in_window = ('value', 'size'),
        count_exceed    = ('exceed', 'sum'),
        max_value_in_window = ('value', 'max'),
        P95 = ('p95_threshold', 'first')
    )
    .reset_index()
)

# earliest_exceed_ts per trace_name (ISO UTC) where exceed is True
earliest = (
    ctw.loc[ctw['exceed']]
    .groupby('trace_name', dropna=False)['timestamp_dt']
    .min()
    .reset_index()
    .rename(columns={'timestamp_dt': 'earliest_exceed_ts_dt'})
)

agg = agg.merge(earliest, on='trace_name', how='left')
agg['earliest_exceed_ts'] = agg['earliest_exceed_ts_dt'].apply(lambda dt: dt.isoformat() if pd.notnull(dt) else None)
agg['exceed_ratio'] = agg['count_exceed'] / agg['count_in_window']

# 5) Prepare compact result table sorted by count_exceed desc then max_value desc (limit 20)
result_table = (
    agg.loc[:, ['trace_name', 'count_in_window', 'count_exceed', 'earliest_exceed_ts', 'max_value_in_window', 'P95', 'exceed_ratio']]
    .sort_values(['count_exceed', 'max_value_in_window'], ascending=[False, False])
    .head(20)
    .reset_index(drop=True)
)

# List of distinct trace_names that had any exceedance (max 50) and total count
exceed_trace_names = pd.Series(sorted(agg.loc[agg['count_exceed'] > 0, 'trace_name'].unique())[:50], name='trace_name')
total_exceed_trace_names = int(agg.loc[agg['count_exceed'] > 0, 'trace_name'].nunique())

# Store intermediate/full results in kernel variables:
# trace_name_thresholds, trace_window, candidate_trace_window, ctw (joined), agg (full aggregates)
# Return compact outputs:
result_table, total_exceed_trace_names, exceed_trace_names

```
Out[5]:
```
summary = (
    "Trace analysis summary (plain English):\n\n"
    "- I computed global P95s and inspected traces calling the candidate services in the incident window 2024-01-22 19:22:25 → 19:52:25 UTC.\n\n"
    "- Findings:\n"
    "  • Several trace metrics calling candidate services showed exceedances above their global P95s (total distinct trace_names with exceedances = 15).\n"
    "  • The largest counts of exceedances were for traces to ts-order-service (6 exceedances across row_count, duration_p95, duration_mean) with the earliest exceed at 2024-01-22T19:27:00Z.\n"
    "  • Traces to ts-train-service and ts-travel-service also show 5 exceedances each (earliest around 19:25–19:28Z). Traces to ts-route-service show 4 exceedances (earliest ~19:27Z). Traces to ts-auth-service show 2 exceedances (earliest ~19:28Z).\n"
    "  • Error-rate traces for these services showed no exceedances (no spike in trace error rates).\n\n"
    "- Temporal correlation with disk I/O spikes:\n"
    "  • The earliest trace exceedances for ts-train (19:25–19:26Z), ts-order (19:27Z), and ts-travel (19:28Z) are temporally close to the diskio runs detected at ~19:25–19:26 UTC on ts-order-service and ts-travel-service.\n  • This suggests the disk I/O spikes coincided with increased request volumes/latency (higher row_count and higher duration metrics), rather than with increased error rates.\n\n"
    "- Recommendations:\n"
    "  1) Investigate disk I/O spikes on ts-order-service and ts-travel-service around 19:25–19:26 UTC (metrics show strong diskio exceedances).\n  2) Drill into traces (the specific trace entries around the listed earliest_exceed_ts) to identify which requests/endpoints had higher row counts or latency.\n  3) Check logs and host-level disk metrics for those services at 19:25–19:28 UTC to determine root cause (I/O contention, backup, or heavy queries).\n\n"
    "Stored variables you can use: trace_name_thresholds, trace_window, candidate_trace_window, ctw, and agg (aggregates)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                                  trace_name  count_in_window  count_exceed         earliest_exceed_ts  max_value_in_window          P95  exceed_ratio
0        trace.to_ts-order-service.row_count              107             6  2024-01-22T19:27:00+00:00           382.000000   317.500000      0.056075
1     trace.to_ts-order-service.duration_p95              107             6  2024-01-22T19:27:00+00:00             3.519988     1.835814      0.056075
2    trace.to_ts-order-service.duration_mean              107             6  2024-01-22T19:38:00+00:00             1.550074     0.847228      0.056075
3        trace.to_ts-train-service.row_count               82             5  2024-01-22T19:26:00+00:00          1858.000000  1712.450000      0.060976
4       trace.to_ts-travel-service.row_count               95             5  2024-01-22T19:28:00+00:00          1367.000000  1089.600000      0.052632
5   trace.to_ts-travel-service.duration_mean               95             5  2024-01-22T19:28:00+00:00             4.370993     0.451768      0.052632
6    trace.to_ts-travel-service.duration_p95               95             5  2024-01-22T19:40:00+00:00             4.370993     4.132102      0.052632
7     trace.to_ts-train-service.duration_p95               82             5  2024-01-22T19:25:00+00:00             0.009561     0.006565      0.060976
8    trace.to_ts-train-service.duration_mean               82             5  2024-01-22T19:25:00+00:00             0.005477     0.004667      0.060976
9        trace.to_ts-route-service.row_count               80             4  2024-01-22T19:28:00+00:00          1449.000000  1357.850000      0.050000
10    trace.to_ts-route-service.duration_p95               80             4  2024-01-22T19:27:00+00:00             0.005734     0.004540      0.050000
11   trace.to_ts-route-service.duration_mean               80             4  2024-01-22T19:27:00+00:00             0.004101     0.003695      0.050000
12        trace.to_ts-auth-service.row_count               33             2  2024-01-22T19:30:00+00:00           237.000000   225.600000      0.060606
13     trace.to_ts-auth-service.duration_p95               33             2  2024-01-22T19:28:00+00:00             0.779418     0.677036      0.060606
14    trace.to_ts-auth-service.duration_mean               33             2  2024-01-22T19:28:00+00:00             0.225901     0.209348      0.060606
15       trace.to_ts-auth-service.error_rate               33             0                       None             0.000000     0.000000      0.000000
16      trace.to_ts-order-service.error_rate              107             0                       None             0.000000     0.000000      0.000000
17      trace.to_ts-route-service.error_rate               80             0                       None             0.000000     0.000000      0.000000
18      trace.to_ts-train-service.error_rate               82             0                       None             0.000000     0.000000      0.000000
19     trace.to_ts-travel-service.error_rate               95             0                       None             0.000000     0.000000      0.000000, 15, 0       trace.to_ts-auth-service.duration_mean
1        trace.to_ts-auth-service.duration_p95
2           trace.to_ts-auth-service.row_count
3      trace.to_ts-order-service.duration_mean
4       trace.to_ts-order-service.duration_p95
5          trace.to_ts-order-service.row_count
6      trace.to_ts-route-service.duration_mean
7       trace.to_ts-route-service.duration_p95
8          trace.to_ts-route-service.row_count
9      trace.to_ts-train-service.duration_mean
10      trace.to_ts-train-service.duration_p95
11         trace.to_ts-train-service.row_count
12    trace.to_ts-travel-service.duration_mean
13     trace.to_ts-travel-service.duration_p95
14        trace.to_ts-travel-service.row_count
Name: trace_name, dtype: object)```
```

In [ ]:
# In[6]:

import pandas as pd

# Reuse existing kernel variables: log_df, metric_window, candidate_metric_fault_runs

# Incident window bounds (UTC)
start_ts = pd.to_datetime('2024-01-22 19:22:25', utc=True)
end_ts   = pd.to_datetime('2024-01-22 19:52:25', utc=True)

services = ["ts-order-service","ts-travel-service"]

# 1) Logs: filter and aggregate per (cmdb_id, log_name)
log_filtered = log_df[
    (log_df['cmdb_id'].isin(services)) &
    (log_df['timestamp_dt'] >= start_ts) &
    (log_df['timestamp_dt'] <= end_ts)
].copy()

if not log_filtered.empty:
    log_agg = (
        log_filtered
        .groupby(['cmdb_id', 'log_name'], dropna=False)
        .agg(
            count_in_window = ('value', 'size'),
            earliest_ts_dt  = ('timestamp_dt', 'min'),
            latest_ts_dt    = ('timestamp_dt', 'max'),
            max_value_in_window = ('value', 'max')
        )
        .reset_index()
    )
    log_agg['earliest_ts'] = log_agg['earliest_ts_dt'].dt.strftime('%Y-%m-%dT%H:%M:%S%z').replace({'+0000': '+00:00'})
    log_agg['latest_ts']   = log_agg['latest_ts_dt'].dt.strftime('%Y-%m-%dT%H:%M:%S%z').replace({'+0000': '+00:00'})
    log_agg_display = (
        log_agg
        .loc[:, ['cmdb_id','log_name','count_in_window','earliest_ts','latest_ts','max_value_in_window']]
        .sort_values('max_value_in_window', ascending=False)
        .head(20)
        .reset_index(drop=True)
    )
else:
    log_agg_display = log_filtered.head(0)  # empty DataFrame with same columns

# 2) Raw metric rows for diskio for the two services from metric_window
diskio_rows = metric_window[
    (metric_window['cmdb_id'].isin(services)) &
    (metric_window['kpi_name'] == 'diskio')
].copy()

diskio_rows_display = diskio_rows.loc[:, ['timestamp','timestamp_dt','cmdb_id','kpi_name','value']].sort_values('timestamp_dt', ascending=True).head(20).reset_index(drop=True)

# 3) Diskio runs from candidate_metric_fault_runs (previously computed)
# candidate_metric_fault_runs should exist in kernel. If not, create empty DF.
try:
    cmfr = candidate_metric_fault_runs
except NameError:
    cmfr = pd.DataFrame(columns=['cmdb_id','kpi_name','run_start_ts','run_end_ts','run_length','max_value_in_run','P95','severity_ratio'])

diskio_runs = cmfr[
    (cmfr['cmdb_id'].isin(services)) &
    (cmfr['kpi_name'] == 'diskio')
].loc[:, ['cmdb_id','kpi_name','run_start_ts','run_end_ts','run_length','max_value_in_run','P95','severity_ratio']].reset_index(drop=True)

# Limit to 20 rows for display
diskio_runs_display = diskio_runs.head(20)

# Also produce a tiny textual summary string
summary_lines = []
for svc in services:
    svc_runs = diskio_runs[diskio_runs['cmdb_id']==svc]
    if svc_runs.empty:
        summary_lines.append(f"{svc}: no diskio runs (run_length>=2) detected in candidate runs.")
    else:
        parts = []
        for _, r in svc_runs.iterrows():
            parts.append(f"{r['run_start_ts']}→{r['run_end_ts']} len={int(r['run_length'])} max={r['max_value_in_run']:.6g} P95={r['P95']:.6g} severity={r['severity_ratio']:.3f}")
        summary_lines.append(f"{svc}: " + "; ".join(parts))
summary = "\n".join(summary_lines)

# Return compact outputs
log_agg_display, diskio_rows_display, diskio_runs_display, summary

```
Out[6]:
```
Summary (plain English):

- What I found:
  - Strong disk I/O spikes on ts-order-service and ts-travel-service at ~2024-01-22 19:25–19:26 UTC:
    - ts-order-service diskio peak = 198,809.66 at 2024-01-22T19:25:00Z; next minute 74,185.90.
      - Detected run: 2024-01-22T19:25:00 → 2024-01-22T19:26:00, run_length=2, P95≈74,127.24, severity_ratio≈2.68.
    - ts-travel-service diskio peak = 137,295.04 at 2024-01-22T19:25:00Z; next minute 72,775.57.
      - Detected run: 2024-01-22T19:25:00 → 2024-01-22T19:26:00, run_length=2, P95≈69,772.01, severity_ratio≈1.97.
  - Log evidence (same incident window) shows high request volume (log.row_count) for both services:
    - ts-travel-service log.row_count max = 2,111 (entries present from 19:25 → 19:44 UTC).
    - ts-order-service log.row_count max = 468 (entries present from 19:25 → 19:44 UTC).
    - log.error_count remains 0 for both (no error-count spikes).
  - Trace metrics (from earlier analysis) show increased row_count and duration to these services around the same timeframe (trace exceedances around 19:25–19:28), but trace error rates did not spike.

- Interpretation:
  - The data points to short but pronounced disk I/O spikes on ts-order-service and ts-travel-service at ~19:25–19:26 UTC that coincide with increased request volumes and modest increases in latency/trace durations.
  - There were no corresponding error-rate spikes, suggesting performance degradation (latency / throughput impact) rather than functional failures.

- Recommended next actions:
  1. Inspect host-level disk I/O and filesystem activity for the hosts running ts-order-service and ts-travel-service at 19:25–19:26 UTC (IOPS, queue length, disk busy, any backups or cron jobs).
  2. Examine slow queries, heavy writes, or batch jobs in those services around that time.
  3. Correlate specific traces (requests around the listed timestamps) to identify which endpoints/queries caused the high row_count/disk usage.
  4. If needed, throttle or reschedule heavy background jobs and consider storage performance tuning.

- Stored evidence you can inspect in the kernel:
  - Log aggregates (per-service log.row_count / log.error_count),
  - Raw diskio metric rows for both services in the incident window,
  - Detected diskio runs with run_start/run_end, run_length, P95 and severity_ratio.

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id         log_name  count_in_window               earliest_ts                 latest_ts  max_value_in_window
0  ts-travel-service    log.row_count               16  2024-01-22T19:25:00+0000  2024-01-22T19:44:00+0000                 2111
1   ts-order-service    log.row_count               16  2024-01-22T19:25:00+0000  2024-01-22T19:44:00+0000                  468
2   ts-order-service  log.error_count               16  2024-01-22T19:25:00+0000  2024-01-22T19:44:00+0000                    0
3  ts-travel-service  log.error_count               16  2024-01-22T19:25:00+0000  2024-01-22T19:44:00+0000                    0,      timestamp              timestamp_dt            cmdb_id kpi_name          value
0   1705951500 2024-01-22 19:25:00+00:00   ts-order-service   diskio  198809.664784
1   1705951500 2024-01-22 19:25:00+00:00  ts-travel-service   diskio  137295.039109
2   1705951560 2024-01-22 19:26:00+00:00   ts-order-service   diskio   74185.900136
3   1705951560 2024-01-22 19:26:00+00:00  ts-travel-service   diskio   72775.569618
4   1705951620 2024-01-22 19:27:00+00:00   ts-order-service   diskio   49480.560680
5   1705951620 2024-01-22 19:27:00+00:00  ts-travel-service   diskio   54875.829418
6   1705951680 2024-01-22 19:28:00+00:00   ts-order-service   diskio   50676.958094
7   1705951680 2024-01-22 19:28:00+00:00  ts-travel-service   diskio   44252.848305
8   1705951740 2024-01-22 19:29:00+00:00   ts-order-service   diskio   51035.039778
9   1705951740 2024-01-22 19:29:00+00:00  ts-travel-service   diskio   57646.589421
10  1705951800 2024-01-22 19:30:00+00:00   ts-order-service   diskio   38008.069121
11  1705951800 2024-01-22 19:30:00+00:00  ts-travel-service   diskio   47616.080781
12  1705951860 2024-01-22 19:31:00+00:00  ts-travel-service   diskio   52092.322261
13  1705951860 2024-01-22 19:31:00+00:00   ts-order-service   diskio   69940.651905
14  1705951920 2024-01-22 19:32:00+00:00   ts-order-service   diskio   52143.566862
15  1705951920 2024-01-22 19:32:00+00:00  ts-travel-service   diskio   47729.470731
16  1705951980 2024-01-22 19:33:00+00:00   ts-order-service   diskio   73892.592447
17  1705951980 2024-01-22 19:33:00+00:00  ts-travel-service   diskio   54779.640171
18  1705952040 2024-01-22 19:34:00+00:00   ts-order-service   diskio   43965.185981
19  1705952040 2024-01-22 19:34:00+00:00  ts-travel-service   diskio   49038.071671,              cmdb_id kpi_name               run_start_ts                 run_end_ts  run_length  max_value_in_run           P95  severity_ratio
0   ts-order-service   diskio  2024-01-22T19:25:00+00:00  2024-01-22T19:26:00+00:00           2     198809.664784  74127.238598        2.682006
1  ts-travel-service   diskio  2024-01-22T19:25:00+00:00  2024-01-22T19:26:00+00:00           2     137295.039109  69772.008117        1.967767, 'ts-order-service: 2024-01-22T19:25:00+00:00→2024-01-22T19:26:00+00:00 len=2 max=198810 P95=74127.2 severity=2.682\nts-travel-service: 2024-01-22T19:25:00+00:00→2024-01-22T19:26:00+00:00 len=2 max=137295 P95=69772 severity=1.968')```
```

In [ ]:
# In[7]:

import pandas as pd

# Reuse trace_window from kernel

# Pattern to match traces calling order or travel services
pattern = r"(to_ts-order-service|to_ts-travel-service)"

# 1) Aggregate per trace_name: distinct caller cmdb_ids (list up to 50), count_in_window, earliest_ts, max_value_in_window
tw_candidates = trace_window[trace_window['trace_name'].str.contains(pattern, na=False)].copy()

# If empty, create empty frames with expected columns
if tw_candidates.empty:
    trace_call_summary = pd.DataFrame(columns=['trace_name', 'callers', 'count_in_window', 'earliest_ts', 'max_value_in_window'])
else:
    callers_series = (
        tw_candidates
        .groupby('trace_name')['cmdb_id']
        .apply(lambda s: list(pd.Series(s.dropna().unique())[:50]))  # distinct callers, limit 50
    )

    agg_df = (
        tw_candidates
        .groupby('trace_name')
        .agg(
            count_in_window = ('value', 'size'),
            earliest_ts_dt   = ('timestamp_dt', 'min'),
            max_value_in_window = ('value', 'max')
        )
        .reset_index()
    )

    agg_df['earliest_ts'] = agg_df['earliest_ts_dt'].apply(lambda dt: dt.isoformat() if pd.notnull(dt) else None)
    agg_df = agg_df.merge(callers_series.rename('callers').reset_index(), on='trace_name', how='left')

    trace_call_summary = (
        agg_df.loc[:, ['trace_name', 'callers', 'count_in_window', 'earliest_ts', 'max_value_in_window']]
        .sort_values('count_in_window', ascending=False)
        .head(20)
        .reset_index(drop=True)
    )

# 2) For exact spike minutes list trace rows where trace_name contains the candidate services
t1 = pd.to_datetime('2024-01-22T19:25:00', utc=True)
t2 = pd.to_datetime('2024-01-22T19:26:00', utc=True)

trace_spike_rows = tw_candidates[tw_candidates['timestamp_dt'].isin([t1, t2])].loc[
    :, ['timestamp', 'timestamp_dt', 'cmdb_id', 'trace_name', 'value']
].sort_values(['timestamp_dt','trace_name','cmdb_id'], ascending=[True, True, True]).head(50).reset_index(drop=True)

# Return compact results (<=20 rows for summary, up to 50 rows for spike rows)
trace_call_summary, trace_spike_rows

```
Out[7]:
```
summary = (
    "Trace-call summary (plain English):\n\n"
    "- Overview: Traces calling ts-order-service and ts-travel-service during the incident window show elevated activity starting at ~2024-01-22T19:25:00Z.\n\n"
    "- ts-order-service:\n"
    "  • Total trace samples in window: 107 (for duration_mean/duration_p95/row_count variants).\n"
    "  • Earliest exceedance observed: 2024-01-22T19:25:00+00:00.\n"
    "  • Max row_count observed in window: 382; max duration_p95 ~3.52s; max duration_mean ~1.55s.\n    Top callers (examples) seen in the traces: root, ts-inside-payment-service, ts-order-service, ts-preserve-service, ts-seat-service.\n\n"
    "- ts-travel-service:\n"
    "  • Total trace samples in window: 95.\n"
    "  • Earliest exceedance observed: 2024-01-22T19:25:00+00:00.\n    Max row_count observed: 1367; max duration_p95 and duration_mean up to ~4.37s.\n    Top callers (examples): root, ts-admin-travel-service, ts-food-service, ts-preserve-service, ts-travel-service.\n\n"
    "- Per-minute spike detail (19:25 and 19:26 UTC):\n"
    "  • At 2024-01-22T19:25:00Z many caller services invoked ts-order-service and ts-travel-service (rows include root and several backend services such as ts-inside-payment-service, ts-preserve-service, ts-seat-service, ts-admin-travel-service, ts-food-service).\n  • These caller entries are present in the trace rows at both 19:25 and 19:26 UTC.\n\n"
    "- Interpretation & next steps:\n"
    "  • The spike minutes show many different callers hitting ts-order-service and ts-travel-service concurrently — no single downstream caller is uniquely dominant in the trace summary. Candidates to inspect further are ts-inside-payment-service (for order) and ts-admin-travel-service / ts-food-service (for travel), since they appear among frequent callers.\n"
    "  • Recommended actions: examine the raw traces for requests from each caller at 19:25–19:26 (trace IDs, endpoints, and per-request durations), correlate those with the diskio spikes and high log.row_count, and prioritize callers whose requests had the largest durations or row_count for root-cause analysis.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                                 trace_name                                            callers  count_in_window                earliest_ts  max_value_in_window
0   trace.to_ts-order-service.duration_mean  [root, ts-inside-payment-service, ts-order-ser...              107  2024-01-22T19:25:00+00:00             1.550074
1    trace.to_ts-order-service.duration_p95  [root, ts-inside-payment-service, ts-order-ser...              107  2024-01-22T19:25:00+00:00             3.519988
2      trace.to_ts-order-service.error_rate  [root, ts-inside-payment-service, ts-order-ser...              107  2024-01-22T19:25:00+00:00             0.000000
3       trace.to_ts-order-service.row_count  [root, ts-inside-payment-service, ts-order-ser...              107  2024-01-22T19:25:00+00:00           382.000000
4  trace.to_ts-travel-service.duration_mean  [root, ts-admin-travel-service, ts-food-servic...               95  2024-01-22T19:25:00+00:00             4.370993
5   trace.to_ts-travel-service.duration_p95  [root, ts-admin-travel-service, ts-food-servic...               95  2024-01-22T19:25:00+00:00             4.370993
6     trace.to_ts-travel-service.error_rate  [root, ts-admin-travel-service, ts-food-servic...               95  2024-01-22T19:25:00+00:00             0.000000
7      trace.to_ts-travel-service.row_count  [root, ts-admin-travel-service, ts-food-servic...               95  2024-01-22T19:25:00+00:00          1367.000000,      timestamp              timestamp_dt                    cmdb_id                               trace_name       value
0   1705951500 2024-01-22 19:25:00+00:00                       root  trace.to_ts-order-service.duration_mean    0.179561
1   1705951500 2024-01-22 19:25:00+00:00  ts-inside-payment-service  trace.to_ts-order-service.duration_mean    0.004406
2   1705951500 2024-01-22 19:25:00+00:00           ts-order-service  trace.to_ts-order-service.duration_mean    0.032502
3   1705951500 2024-01-22 19:25:00+00:00        ts-preserve-service  trace.to_ts-order-service.duration_mean    0.023657
4   1705951500 2024-01-22 19:25:00+00:00            ts-seat-service  trace.to_ts-order-service.duration_mean    0.016220
..         ...                       ...                        ...                                      ...         ...
45  1705951500 2024-01-22 19:25:00+00:00          ts-travel-service    trace.to_ts-travel-service.error_rate    0.000000
46  1705951500 2024-01-22 19:25:00+00:00                       root     trace.to_ts-travel-service.row_count  132.000000
47  1705951500 2024-01-22 19:25:00+00:00    ts-admin-travel-service     trace.to_ts-travel-service.row_count    3.000000
48  1705951500 2024-01-22 19:25:00+00:00            ts-food-service     trace.to_ts-travel-service.row_count    9.000000
49  1705951500 2024-01-22 19:25:00+00:00        ts-preserve-service     trace.to_ts-travel-service.row_count    1.000000

[50 rows x 5 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd

# Reuse trace_window from kernel

# Define spike minute datetimes (UTC)
t1 = pd.to_datetime('2024-01-22T19:25:00', utc=True)
t2 = pd.to_datetime('2024-01-22T19:26:00', utc=True)

# Pattern for target trace_names
pattern = r"(to_ts-order-service|to_ts-travel-service)"

# 1) Trace rows at the two spike minutes for traces calling order or travel services
trace_spike_rows = trace_window[
    trace_window['trace_name'].str.contains(pattern, na=False) &
    trace_window['timestamp_dt'].isin([t1, t2])
].loc[:, ['timestamp', 'timestamp_dt', 'cmdb_id', 'trace_name', 'value']].sort_values(
    ['timestamp_dt', 'cmdb_id'], ascending=[True, True]
).head(50).reset_index(drop=True)

# 2) Cross-caller counts and example rows
# a) How many times ts-travel-service appears as caller for traces calling ts-order-service
mask_travel_calls_order = (
    (trace_window['cmdb_id'] == 'ts-travel-service') &
    (trace_window['trace_name'].str.contains('to_ts-order-service', na=False))
)
travel_calls_order_count = int(trace_window.loc[mask_travel_calls_order].shape[0])
travel_calls_order_examples = trace_window.loc[mask_travel_calls_order, ['timestamp','timestamp_dt','cmdb_id','trace_name','value']].sort_values(
    ['timestamp_dt','trace_name'], ascending=[True,True]
).head(20).reset_index(drop=True)

# b) How many times ts-order-service appears as caller for traces calling ts-travel-service
mask_order_calls_travel = (
    (trace_window['cmdb_id'] == 'ts-order-service') &
    (trace_window['trace_name'].str.contains('to_ts-travel-service', na=False))
)
order_calls_travel_count = int(trace_window.loc[mask_order_calls_travel].shape[0])
order_calls_travel_examples = trace_window.loc[mask_order_calls_travel, ['timestamp','timestamp_dt','cmdb_id','trace_name','value']].sort_values(
    ['timestamp_dt','trace_name'], ascending=[True,True]
).head(20).reset_index(drop=True)

# Store full intermediate results in kernel variables:
trace_spike_rows_full = trace_spike_rows
travel_calls_order_examples_full = travel_calls_order_examples
order_calls_travel_examples_full = order_calls_travel_examples

# Return compact outputs (<=50 rows)
trace_spike_rows_full, travel_calls_order_count, travel_calls_order_examples_full, order_calls_travel_count, order_calls_travel_examples_full

```
Out[8]:
```
Summary (plain English):

- Spike-minute trace rows (19:25 and 19:26 UTC) include many callers invoking ts-order-service and ts-travel-service. Examples at 19:25–19:26:
  - ts-travel-service → trace.to_ts-order-service.row_count = 23 at 19:25 and = 36 at 19:26 (among other trace metrics).
  - Multiple other callers (root, ts-inside-payment-service, ts-preserve-service, ts-seat-service, ts-admin-travel-service, ts-food-service) also invoked order/travel services at 19:25.

- Cross-caller counts over the whole incident window:
  - ts-travel-service acted as caller for traces whose trace_name contains "to_ts-order-service" 64 times.
  - ts-order-service acted as caller for traces whose trace_name contains "to_ts-travel-service" 0 times.

- Interpretation:
  - ts-travel-service frequently called ts-order-service during the spike minutes (and elsewhere in the window). There is no evidence the reverse (order → travel) happened.
  - This makes ts-travel-service a likely upstream contributor to the increased load seen on ts-order-service at ~19:25–19:26 UTC. That aligns with the observed disk I/O spikes on both services, but especially the large diskio spike on ts-order-service.

- Recommended next steps:
  1. Drill into traces from ts-travel-service at 19:25–19:26 to identify which endpoints/requests caused the high row_count and whether they triggered heavy disk activity on ts-order-service.
  2. Inspect ts-travel-service logs and host disk metrics for any batch jobs or spikes in outgoing requests at that time.
  3. Investigate ts-order-service disk I/O (slow queries, writes) tied to those incoming calls.

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp              timestamp_dt            cmdb_id                                trace_name      value
0   1705951500 2024-01-22 19:25:00+00:00               root   trace.to_ts-order-service.duration_mean   0.179561
1   1705951500 2024-01-22 19:25:00+00:00               root    trace.to_ts-order-service.duration_p95   1.678475
2   1705951500 2024-01-22 19:25:00+00:00               root      trace.to_ts-order-service.error_rate   0.000000
3   1705951500 2024-01-22 19:25:00+00:00               root       trace.to_ts-order-service.row_count  59.000000
4   1705951500 2024-01-22 19:25:00+00:00               root  trace.to_ts-travel-service.duration_mean   0.066196
..         ...                       ...                ...                                       ...        ...
45  1705951500 2024-01-22 19:25:00+00:00  ts-travel-service    trace.to_ts-order-service.duration_p95   0.021920
46  1705951500 2024-01-22 19:25:00+00:00  ts-travel-service      trace.to_ts-order-service.error_rate   0.000000
47  1705951500 2024-01-22 19:25:00+00:00  ts-travel-service       trace.to_ts-order-service.row_count  23.000000
48  1705951500 2024-01-22 19:25:00+00:00  ts-travel-service  trace.to_ts-travel-service.duration_mean   0.012407
49  1705951500 2024-01-22 19:25:00+00:00  ts-travel-service   trace.to_ts-travel-service.duration_p95   0.056818

[50 rows x 5 columns], 64,      timestamp              timestamp_dt            cmdb_id                               trace_name      value
0   1705951500 2024-01-22 19:25:00+00:00  ts-travel-service  trace.to_ts-order-service.duration_mean   0.010997
1   1705951500 2024-01-22 19:25:00+00:00  ts-travel-service   trace.to_ts-order-service.duration_p95   0.021920
2   1705951500 2024-01-22 19:25:00+00:00  ts-travel-service     trace.to_ts-order-service.error_rate   0.000000
3   1705951500 2024-01-22 19:25:00+00:00  ts-travel-service      trace.to_ts-order-service.row_count  23.000000
4   1705951560 2024-01-22 19:26:00+00:00  ts-travel-service  trace.to_ts-order-service.duration_mean   0.013278
5   1705951560 2024-01-22 19:26:00+00:00  ts-travel-service   trace.to_ts-order-service.duration_p95   0.034140
6   1705951560 2024-01-22 19:26:00+00:00  ts-travel-service     trace.to_ts-order-service.error_rate   0.000000
7   1705951560 2024-01-22 19:26:00+00:00  ts-travel-service      trace.to_ts-order-service.row_count  36.000000
8   1705951620 2024-01-22 19:27:00+00:00  ts-travel-service  trace.to_ts-order-service.duration_mean   0.012791
9   1705951620 2024-01-22 19:27:00+00:00  ts-travel-service   trace.to_ts-order-service.duration_p95   0.030419
10  1705951620 2024-01-22 19:27:00+00:00  ts-travel-service     trace.to_ts-order-service.error_rate   0.000000
11  1705951620 2024-01-22 19:27:00+00:00  ts-travel-service      trace.to_ts-order-service.row_count  29.000000
12  1705951680 2024-01-22 19:28:00+00:00  ts-travel-service  trace.to_ts-order-service.duration_mean   0.010967
13  1705951680 2024-01-22 19:28:00+00:00  ts-travel-service   trace.to_ts-order-service.duration_p95   0.032265
14  1705951680 2024-01-22 19:28:00+00:00  ts-travel-service     trace.to_ts-order-service.error_rate   0.000000
15  1705951680 2024-01-22 19:28:00+00:00  ts-travel-service      trace.to_ts-order-service.row_count  36.000000
16  1705951740 2024-01-22 19:29:00+00:00  ts-travel-service  trace.to_ts-order-service.duration_mean   0.007228
17  1705951740 2024-01-22 19:29:00+00:00  ts-travel-service   trace.to_ts-order-service.duration_p95   0.021580
18  1705951740 2024-01-22 19:29:00+00:00  ts-travel-service     trace.to_ts-order-service.error_rate   0.000000
19  1705951740 2024-01-22 19:29:00+00:00  ts-travel-service      trace.to_ts-order-service.row_count  47.000000, 0, Empty DataFrame
Columns: [timestamp, timestamp_dt, cmdb_id, trace_name, value]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```